# Waterfall and combination charts

> **Krisk 0.9 notebook mode**
>
> This notebook runs locally with pandas and Krisk only. It does not require the Krisk
> server, MCP, SQLite, or PostgreSQL. Chart output embeds its renderer and can be used
> offline. The typed `ChartSpec` API is recommended for new work; legacy `krisk.plot`
> calls remain supported for existing notebooks.

Waterfall charts explain cumulative changes. Combination charts place bar and
line series on one category axis. Both are available through `ChartSpec`; historical
helpers remain available for existing notebooks.

In [ ]:
from pathlib import Path

import pandas as pd
import krisk.plot as kk
from krisk import Chart, ChartSpec, __version__

ROOT = Path.cwd()
if not (ROOT / "krisk" / "tests" / "data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "krisk" / "tests" / "data"

print(f"Krisk {__version__}")
import numpy as np

## Waterfall

In [ ]:
changes = pd.DataFrame(
    {
        "stage": ["Opening", "Sales", "Returns", "Costs", "Closing"],
        "change": [120, 48, -9, -31, 22],
    }
)

Chart.from_dataframe(
    changes,
    ChartSpec(
        kind="waterfall",
        x="stage",
        y="change",
        title="Contribution to closing value",
        annotate=True,
    ),
)

The legacy helper can color-code increases and decreases.

In [ ]:
kk.waterfall(
    changes["change"],
    color_coded=True,
    annotate="outside",
    up_name="increase",
)

## Bar and line combination

In [ ]:
gapminder = pd.read_csv(DATA_DIR / "gapminderDataFiveYear.txt", sep="\t")
latest = gapminder[gapminder.year == 2007]

In [ ]:
Chart.from_dataframe(
    latest,
    ChartSpec(
        kind="combo",
        x="continent",
        y="lifeExp",
        y2="gdpPercap",
        aggregate="mean",
        aggregate_y2="mean",
        title="Life expectancy and income in 2007",
        description="Bars show life expectancy; the line shows GDP per capita",
    ),
)

The equivalent compatibility helper is `bar_line`.

In [ ]:
kk.bar_line(latest, "continent", "lifeExp", "gdpPercap")

Set `is_distinct=True` only when every category already has exactly one bar and one
line value. Otherwise keep aggregation enabled so duplicate categories are handled
explicitly.

In [ ]:
continent_means = latest.groupby("continent", as_index=False)[
    ["lifeExp", "gdpPercap"]
].mean()
kk.bar_line(
    continent_means,
    "continent",
    "lifeExp",
    "gdpPercap",
    is_distinct=True,
)